# 🏡 Ames 집값 예측: 경사하강법으로 선형회귀

미국 아이오와주 에임스(Ames)에서 실제로 거래된 집 **1,460채**의 데이터입니다. 한 행이 **집 한 채**예요.

| 변수 | 뜻 |
| --- | --- |
| `GrLivArea` | 지상층 생활 면적 |
| `TotalBsmtSF` | 지하층 면적 |
| `GarageArea` | 차고 면적 |
| `YearBuilt` | 건축 연도 |
| `SalePrice` | 판매 가격 (**예측할 정답**) |

순서: 데이터 보기 → 변수 1개로 회귀 → 변수 4개로 회귀 → 비교하고 해석하기

### ✏️ 학생용 활동 안내

코드 곳곳에 **`______` 빈칸**이 있어요 (모두 21개).
1. 빈칸을 채운 뒤 셀을 위에서부터 차례로 실행하세요 (Shift + Enter).
2. 빈칸을 채우지 않으면 오류가 나요. 오류가 나면 빈칸을 다시 확인하세요!
3. 막히면 각 셀 아래의 **🔍 코드 한 줄씩 보기**와 위쪽 설명을 힌트로 활용하세요.

## 1. 데이터 불러오기

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

house = fetch_openml(name='house_prices', as_frame=True).frame
print('데이터 크기:', house.shape)   # (집 수, 변수 수)

데이터 크기: (1460, 81)


## 2. 필요한 변수만 고르고 단위 바꾸기

- 면적: 제곱피트 → **㎡** (1제곱피트 ≈ 0.0929㎡, 참고로 1평 ≈ 3.3㎡)
- 가격: 달러 → **만 달러**
- 생활 면적이 4,000제곱피트(약 372㎡)가 넘는 특수 거래 몇 건은 제외합니다.

In [ ]:
# ✏️ 빈칸 2개: ______ 를 채운 뒤 실행하세요
df = house[house['GrLivArea'] < 4000].copy()     # 특수 거래(이상치) 제외

data = pd.DataFrame({
    'area':     df['GrLivArea'] * ______,        # 생활 면적 (㎡)
    'basement': df['TotalBsmtSF'] * 0.0929,      # 지하 면적 (㎡)
    'garage':   df['GarageArea'] * 0.0929,       # 차고 면적 (㎡)
    'year':     df['YearBuilt'],                 # 건축 연도
    'price':    df['SalePrice'] / ______,         # 가격 (만 달러)
}).dropna()

print('사용할 집 수:', len(data))
data.describe().round(1)

## 3. 산점도로 관계 살펴보기

❓ 어떤 변수가 가격과 가장 관계가 깊어 보이나요?

In [ ]:
# ✏️ 빈칸 1개: ______ 를 채운 뒤 실행하세요
features = ['area', 'basement', 'garage', 'year']

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for i, f in enumerate(features):
    ax[i].scatter(data[f], data[______], s=5, alpha=0.4)
    ax[i].set_xlabel(f)
    ax[i].set_title(f'{f}  (r = {data[f].corr(data["price"]):.2f})')
ax[0].set_ylabel('price (10k USD)')
plt.show()

## 4. 변수 1개로 회귀: 면적 → 가격

$$\hat{y} = a \cdot \text{면적} + b$$

면적은 100~300 정도로 숫자가 커서, 그대로 학습하면 절편 b를 찾는 데 시간이 아주 오래 걸려요.
그래서 **면적에서 평균만 빼서**(중심화) 학습합니다.

- 평균을 빼는 건 그래프를 옆으로 옮기는 것뿐이라 **기울기 a는 그대로**예요. 단위도 그대로 ㎡!
- 절편만 마지막에 원래 위치로 되돌리면 됩니다.
- 면적은 숫자가 커서 기울기 a의 보폭(학습률)은 작게, 절편 b의 보폭은 크게 따로 정해 줍니다.

In [ ]:
# ✏️ 빈칸 6개: ______ 를 채운 뒤 실행하세요
x = data['area'].values
y = data['price'].values

x_mean = x.______()
xc = x - x_mean                   # 중심화: 평균만 빼기 (단위는 그대로 ㎡)

a, b = 0.0, 0.0
lr_a = 0.0002                     # 기울기용 보폭 (면적 숫자가 커서 작게)
lr_b = 0.1                        # 절편용 보폭
epochs = 300
loss1 = []

for epoch in range(epochs):
    y_hat = ______
    error = ______
    loss1.append(np.mean(error ______ 2))
    a -= lr_a * (-2 * np.mean(xc * error))
    b -= lr_b * (-2 * np.mean(______))

a1 = a                            # 기울기는 그대로
b1 = b - ______ * x_mean               # 절편만 원래 위치로 되돌리기
print(f'가격(만 달러) = {a1:.4f} × 면적(㎡) + {b1:.2f}')
print(f'→ 면적이 1㎡ 늘면 가격이 약 {a1 * 10000:,.0f}달러 오른다')

In [ ]:
# ✏️ 빈칸 1개: ______ 를 채운 뒤 실행하세요
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].plot(loss1)
ax[0].set_xlabel('epoch')
ax[0].set_ylabel('MSE')
ax[0].set_title('Loss decreases')

ax[1].scatter(x, y, s=5, alpha=0.4)
xs = np.linspace(x.min(), x.max(), 100)
ax[1].plot(xs, ______, color='red', linewidth=2)
ax[1].set_xlabel('area (m²)')
ax[1].set_ylabel('price (10k USD)')
ax[1].set_title(f'price = {a1:.3f} × area + {b1:.2f}')
plt.show()

## 5. 변수 4개로 회귀: 다중 선형회귀

$$\hat{y} = w_1 \cdot \text{면적} + w_2 \cdot \text{지하} + w_3 \cdot \text{차고} + w_4 \cdot \text{연도} + b$$

변수 1개일 때와 똑같은 방법이에요. 달라지는 건 기울기가 **4개**가 된다는 것뿐!
`Xc @ w`는 각 집마다 $w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4$를 한 번에 계산합니다.

해석: **다른 변수는 그대로 두고** 이 변수만 1(㎡ 또는 1년) 늘면 가격이 $w$만 달러 변한다.

In [ ]:
# ✏️ 빈칸 6개: ______ 를 채운 뒤 실행하세요
X = data[features].values            # (집 수, 4) 행렬, 원래 단위
X_mean = X.mean(axis=0)
Xc = X - ______                      # 변수마다 평균만 빼기 (단위 그대로)

n, d = Xc.shape
w = np.zeros(d)                      # 기울기 4개
b = 0.0
lr_w = 0.0001                        # 기울기용 보폭 (변수 숫자가 커서 작게)
lr_b = 0.1                           # 절편용 보폭
epochs = 3000
loss4 = []

for epoch in range(epochs):
    y_hat = Xc ______ w + b
    error = ______
    loss4.append(np.mean(error ______ 2))
    w -= lr_w * (-2 * Xc.T @ error / n)  # 기울기 4개를 한 번에 미분
    b -= lr_b * (-2 * np.mean(______))

b4 = b - np.sum(w * ______)          # 절편만 원래 위치로 되돌리기

units = {'area': '㎡', 'basement': '㎡', 'garage': '㎡', 'year': '년'}
print('계수 (원래 단위)')
for f, wi in zip(features, w):
    print(f'{f:>9}: {wi:+.4f}   → {f} 1{units[f]} 늘면 약 {wi * 10000:+,.0f}달러')
print(f'{"절편":>9}: {b4:+.2f}')

## 6. 변수 1개 vs 4개 비교하기

**R² (결정계수)**: 1에 가까울수록 데이터를 잘 설명해요.

In [ ]:
# ✏️ 빈칸 2개: ______ 를 채운 뒤 실행하세요
def r2(y, y_hat):
    return 1 - np.sum((y - y_hat) ** 2) / np.sum((y - ______) ** 2)

y_hat1 = a1 * x + b1
y_hat4 = X @ w + ______

print(f'변수 1개 (면적)              MSE = {np.mean((y - y_hat1)**2):7.2f}   R² = {r2(y, y_hat1):.3f}')
print(f'변수 4개 (면적·지하·차고·연도) MSE = {np.mean((y - y_hat4)**2):7.2f}   R² = {r2(y, y_hat4):.3f}')

# 실제 가격 vs 예측 가격: 대각선에 가까울수록 정확
plt.figure(figsize=(5.5, 5))
plt.scatter(y, y_hat4, s=5, alpha=0.4)
lim = [0, y.max()]
plt.plot(lim, lim, color='red')
plt.xlabel('actual price (10k USD)')
plt.ylabel('predicted price (10k USD)')
plt.title('4 variables: actual vs predicted')
plt.show()

## 7. scikit-learn, statsmodels와 비교하기

세 방법 모두 **같은 원래 단위 데이터**를 넣었으니 계수가 같게 나와야 해요.

In [ ]:
# ✏️ 빈칸 1개: ______ 를 채운 뒤 실행하세요
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf

sk = LinearRegression().fit(X, y)
lm = smf.ols('price ~ ______', data=data).fit()

print('변수        경사하강법    scikit-learn   statsmodels')
for i, f in enumerate(features):
    print(f'{f:>9}:  {w[i]:+10.4f}    {sk.coef_[i]:+10.4f}    {lm.params[f]:+10.4f}')
print(f'{"절편":>9}:  {b4:+10.2f}    {sk.intercept_:+10.2f}    {lm.params["Intercept"]:+10.2f}')

### 통계적 해석 (R의 lm과 같은 결과표)

- `coef`: 계수 · `P>|t|`: p값 (0.05보다 작으면 의미 있는 변수) · `[0.025 0.975]`: 95% 신뢰구간
- `R-squared`: 6번에서 직접 계산한 R²와 같은 값

In [ ]:
print(lm.summary())

## 8. 🏠 우리 집 가격 예측해 보기

숫자를 바꿔 가며 예측해 보세요. (30평 ≈ 99㎡)

In [ ]:
# ✏️ 빈칸 2개: ______ 를 채운 뒤 실행하세요
my_area = 150      #@param {type:"number"}
my_basement = 80   #@param {type:"number"}
my_garage = 40     #@param {type:"number"}
my_year = 2000     #@param {type:"number"}

my_house = np.array([my_area, my_basement, my_garage, my_year])
price = my_house @ ______ + ______                    # 원래 단위 그대로 넣으면 끝!
print(f'예측 가격: 약 {price:.1f}만 달러 (약 {price * 10000:,.0f}달러)')